# K0 — Pareto-Optimality Check: Does SSRA Keep MedQA Accuracy?

**OOM fixes applied (vs original notebook):**
| Fix | Effect |
|---|---|
| `PYTORCH_ALLOC_CONF=expandable_segments:True` before any CUDA call | Eliminates allocator fragmentation OOM |
| `enable_input_require_grads()` replaces `prepare_model_for_kbit_training()` | Eliminates bfloat16→float32 upcast (~3.8 GB saved) |
| `device_map={"":0}` instead of `"auto"` | Pins model to GPU 0, prevents split-GPU fragmentation |
| `per_device_train_batch_size` 4→1, `gradient_accumulation_steps` 4→16 | Effective batch unchanged at 16; peak activations cut ~75% |
| `gradient_checkpointing=True` | ~30% activation memory saving |
| `max_seq_length` 1024→512 | Halves KV-cache footprint per batch |
| Stronger `free()` with `synchronize()` + `reset_peak_memory_stats()` | Full CUDA flush between experiments |

In [1]:
# ── Cell 0: Allocator config — MUST run before any CUDA import ───────────────
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("✓ PYTORCH_ALLOC_CONF=expandable_segments:True")

✓ PYTORCH_ALLOC_CONF=expandable_segments:True


In [2]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q --upgrade \
    "transformers>=4.51.0" \
    "peft>=0.14.0" \
    "trl>=0.12.0" \
    "accelerate>=1.2.0" \
    "bitsandbytes>=0.44.1" \
    "datasets>=3.0.2" \
    "sentencepiece" \
    "scipy" \
    "huggingface_hub>=0.23.0" 2>&1 | tail -8

# Required for Qwen3.5-9B hybrid (DeltaNet) fast path — without these
# transformers falls back to slow O(seq_len) torch implementation,
# making training ~10-50x slower on T4.
!pip install -q causal-conv1d
!pip install -q git+https://github.com/fla-org/flash-linear-attention.git

print("✓ install done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 78.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  error: subprocess-exited-with-error
  
  × Building wheel for causal-conv1d (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for causal-conv1d
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (causal-conv1d)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


✓ install done


In [3]:
# ── Cell 2: HF authentication (optional for public Qwen3.5-9B) ───────────────
import os
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf_token
    from huggingface_hub import login
    login(token=_hf_token, add_to_git_credential=False)
    print("✓ Logged in to HuggingFace Hub")
except Exception as e:
    print(f"HF_TOKEN not found ({e}). Proceeding without login — OK for Qwen/Qwen3.5-9B.")

HF_TOKEN not found (Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 117555661 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.). Proceeding without login — OK for Qwen/Qwen3.5-9B.


In [4]:
# ── Cell 3: Imports & global config ──────────────────────────────────────────
import os, json, gc, random, re
from pathlib import Path
import numpy as np, pandas as pd, torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, __version__ as tf_ver
)
from peft import LoraConfig, get_peft_model
# NOTE: prepare_model_for_kbit_training intentionally NOT imported.
# It casts bfloat16 params to float32 (peft/utils/other.py line ~186),
# which caused the observed 3.79 GiB OutOfMemoryError.
# Replacement: model.enable_input_require_grads() — same effect, no dtype cast.
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset
import matplotlib.pyplot as plt

WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
print(f"device={device}  VRAM={vram_gb} GB  transformers={tf_ver}")
assert torch.cuda.is_available(), "GPU is required — enable GPU accelerator in Kaggle settings."
assert vram_gb >= 14, f"Need ≥14 GB VRAM, got {vram_gb} GB. Use T4 or P100 accelerator."

device=cuda  VRAM=15.6 GB  transformers=5.7.0


In [5]:
# ── Cell 4: Model constants & helpers ────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-7B"   # switched from Qwen3.5-9B: pure transformer, no DeltaNet fast-path dependency
RANK     = 16
STEPS    = 500
SEED     = 42
N_TEST   = 200
N_REPLAY = 100
LETTERS  = ["A", "B", "C", "D", "E"]

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

def load_base():
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=BNB,
        device_map={"":0},
        dtype=torch.bfloat16,
    )
    model.config.use_cache = False
    return tok, model

def attach_lora(model, r):
    model.enable_input_require_grads()

    cfg = LoraConfig(
        r=r,
        lora_alpha=r,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model

def free():
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(f"✓ Config loaded. Model: {MODEL_ID}")

✓ Config loaded. Model: Qwen/Qwen2.5-7B


In [6]:
# ── Cell 5: Load datasets ─────────────────────────────────────────────────────
MEDQA_DS = "GBaker/MedQA-USMLE-4-options-hf"

# MedQA questions can exceed 512 tokens when options are long.
# 400 chars for the question leaves ~1100 chars for options+answer, keeping
# total well under 512 tokens (Qwen tokenizer ~1.5-2 chars/token).
_MAX_Q_CHARS = 1200

def fmt_medqa(ex):
    q = ex["sent1"][:_MAX_Q_CHARS]
    opts = [ex["ending0"], ex["ending1"], ex["ending2"], ex["ending3"]]
    opt_text = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(opts))
    ans = LETTERS[int(ex["label"])]
    return {"text": f"Question: {q}\n{opt_text}\nAnswer: {ans}"}

print("Loading MedQA (GBaker parquet) ...")
medqa_train_raw = load_dataset(MEDQA_DS, split="train").select(range(4096))
medqa_train_ds  = medqa_train_raw.map(fmt_medqa, remove_columns=medqa_train_raw.column_names)
medqa_test      = load_dataset(MEDQA_DS, split="test").select(range(N_TEST))
print(f"MedQA  train={len(medqa_train_ds)}  test={len(medqa_test)}")
print(f"Sample: {medqa_train_ds[0]['text'][:200]}")

print("\nStreaming 100 C4 examples ...")
c4 = load_dataset("allenai/c4", "en", split="train", streaming=True, trust_remote_code=True)
real = [{"text": ex["text"][:1000]} for i, ex in zip(range(N_REPLAY), c4)]
real_ds = Dataset.from_list(real)
print(f"Real replay buffer: {len(real_ds)} examples")

Loading MedQA (GBaker parquet) ...


README.md:   0%|          | 0.00/640 [00:00<?, ?B/s]

train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

...

MedQA  train=4096  test=200
Sample: Question: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranber

Streaming 100 C4 examples ...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Real replay buffer: 100 examples


In [7]:
# ── Cell 6: Generate endogenous (SSRA) anchor buffer ─────────────────────────
# If you have uploaded endogenous_anchor_buffer.jsonl as a Kaggle dataset,
# this cell loads it directly and skips the ~27-minute generation step.

ANCHOR_PROMPTS = [
    "Explain Newton's second law with a worked example.",
    "What is the chain rule in calculus, with one example?",
    "Describe how DNA polymerase works during replication.",
    "Solve x^2 + 5x + 6 = 0. Show your work.",
    "Summarise the causes of World War I in three sentences.",
    "What is the main argument of Hobbes' Leviathan?",
    "Compare Buddhism and Stoicism on suffering.",
    "Who wrote Pride and Prejudice and what is its plot?",
    "What should you do if you smell smoke at home?",
    "If a glass falls off a table, what happens and why?",
    "Write a Python function that returns the n-th Fibonacci number.",
    "Explain what a hash map is in two sentences.",
    "Name the seven continents.",
    "How is the U.S. president elected?",
    "What is BMI and how is it computed?",
    "Why does washing hands prevent disease?",
]

# Paths to check — add your dataset slug here if it differs
_CANDIDATE_PATHS = [
    Path("/kaggle/input/datasets/dj4242/ssra-anchor-buffer/endogenous_anchor_buffer.jsonl"),
    WORK / "endogenous_anchor_buffer.jsonl",
]

_prebuilt = next((p for p in _CANDIDATE_PATHS if p.exists()), None)

if _prebuilt is not None:
    print(f"✓ Loading pre-built anchor buffer from {_prebuilt}")
    endo_ds = Dataset.from_json(str(_prebuilt))
    # Copy to /kaggle/working so downstream cells can always find it there
    if _prebuilt != WORK / "endogenous_anchor_buffer.jsonl":
        import shutil
        shutil.copy2(_prebuilt, WORK / "endogenous_anchor_buffer.jsonl")
    print(f"✓ Endogenous anchor buffer: {len(endo_ds)} examples (skipped generation)")
else:
    print(">>> No pre-built buffer found — generating anchors with base model (~27 min) ...")
    set_seed(SEED)
    tok, base = load_base()
    print(f"GPU after load: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

    prompts = (ANCHOR_PROMPTS * (N_REPLAY // len(ANCHOR_PROMPTS) + 1))[:N_REPLAY]
    endo = []
    _think_re = re.compile(r"<think>.*?</think>", re.DOTALL)

    for i, p in enumerate(prompts):
        ids = tok(p, return_tensors="pt").to(base.device)
        with torch.no_grad():
            gen = base.generate(
                **ids,
                max_new_tokens=128,
                do_sample=True,
                temperature=0.7,
                top_p=0.8,
                top_k=20,
                pad_token_id=tok.pad_token_id,
            )
        decoded = tok.decode(gen[0], skip_special_tokens=True)
        cleaned = _think_re.sub("", decoded).strip()
        endo.append({"text": cleaned})
        if (i + 1) % 20 == 0:
            print(f"  generated {i+1}/{N_REPLAY} anchors")

    endo_ds = Dataset.from_list(endo)
    print(f"✓ Endogenous anchor buffer: {len(endo_ds)} examples")
    endo_ds.to_json(WORK / "endogenous_anchor_buffer.jsonl")
    print("Saved to /kaggle/working/endogenous_anchor_buffer.jsonl")

    del base
    free()
    print(f"GPU after anchor cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

✓ Loading pre-built anchor buffer from /kaggle/input/datasets/dj4242/ssra-anchor-buffer/endogenous_anchor_buffer.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

✓ Endogenous anchor buffer: 100 examples (skipped generation)


In [8]:
# ── Cell 7: MedQA accuracy evaluator ─────────────────────────────────────────
def medqa_acc(model, tok):
    model.eval()
    correct = 0
    for ex in medqa_test:
        keys     = ["A", "B", "C", "D"]
        opts     = [ex["ending0"], ex["ending1"], ex["ending2"], ex["ending3"]]
        opt_text = "\n".join(f"{k}. {v}" for k, v in zip(keys, opts))
        prompt   = f"Question: {ex['sent1']}\n{opt_text}\nAnswer:"

        ids = tok(
            prompt, return_tensors="pt",
            truncation=True, max_length=512    # FIX: matches training max_seq_length
        ).to(model.device)

        with torch.no_grad():
            logits = model(**ids).logits[0, -1]

        letter_ids = [
            tok(" " + L, add_special_tokens=False)["input_ids"][-1]
            for L in keys
        ]
        pred_idx = int(logits[letter_ids].argmax().item())
        gold_idx = int(ex["label"])
        correct += int(pred_idx == gold_idx)

    return correct / len(medqa_test)

print("✓ medqa_acc() defined (GBaker schema)")

✓ medqa_acc() defined (GBaker schema)


In [9]:
# ── Cell 8: Train-and-score loop ──────────────────────────────────────────────
def train_and_score(label, train_data):
    print(f"\n{'='*55}\n  {label}\n{'='*55}")
    set_seed(SEED)
    tok, model = load_base()
    tok.model_max_length = 512   # FIX: max_seq_length removed from SFTConfig in TRL>=0.17; set on tokenizer instead
    model = attach_lora(model, RANK)
    print(f"GPU after attach_lora: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    args = SFTConfig(
        output_dir=f"/tmp/{label}",
        # FIX: batch 4->1, grad-accum 4->16 keeps effective batch=16
        # while reducing peak activation memory by ~75%.
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        # FIX: gradient checkpointing trades one recompute pass for ~30% less
        # activation memory — essential fitting a 9B model in 16 GB.
        gradient_checkpointing=True,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        max_steps=STEPS,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        bf16=True,
        dataset_text_field="text",
        dataset_num_proc=1,
        dataloader_num_workers=0,
        seed=SEED,
    )

    trainer = SFTTrainer(
        model=model,
        processing_class=tok,
        train_dataset=train_data,
        args=args,
    )
    trainer.train()

    print(f"\nEvaluating MedQA accuracy for {label} ...")
    acc = medqa_acc(model, tok)
    print(f"✓ {label}: MedQA test accuracy = {acc:.4f}  ({acc*100:.1f}%)")

    # FIX: delete trainer first (releases optimizer states), then model
    del trainer
    del model
    free()
    print(f"GPU after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")
    return acc


# ── E02: no replay ────────────────────────────────────────────────────────────
acc_E02 = train_and_score("E02_noreplay", medqa_train_ds)

# ── E10: 100 real (C4) replay mixed in ───────────────────────────────────────
acc_E10 = train_and_score(
    "E10_real100",
    Dataset.from_list(list(medqa_train_ds) + list(real_ds)).shuffle(seed=SEED)
)

# ── E11: 100 endogenous (SSRA) replay mixed in ────────────────────────────────
acc_E11 = train_and_score(
    "E11_ssra100",
    Dataset.from_list(list(medqa_train_ds) + list(endo_ds)).shuffle(seed=SEED)
)


  E02_noreplay


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Adding EOS to train dataset (num_proc=1):   0%|          | 0/4096 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/4096 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
50,2.801707
100,2.624901
150,2.542355
200,2.435732


In [ ]:
# ── Cell 9: Pareto table ──────────────────────────────────────────────────────
mmlu_forgetting = {
    "E02_noreplay" : 0.352,
    "E10_real100"  : 0.142,
    "E11_ssra100"  : 0.070,
}
medqa_scores = {
    "E02_noreplay" : acc_E02,
    "E10_real100"  : acc_E10,
    "E11_ssra100"  : acc_E11,
}

df = pd.DataFrame([
    {"condition": k, "MedQA_acc": medqa_scores[k], "MMLU_forgetting": mmlu_forgetting[k]}
    for k in mmlu_forgetting
])
print(df.to_string(index=False))
df.to_csv(WORK / "k0_pareto_table.csv", index=False)

delta_medqa      = acc_E11 - acc_E02
delta_forgetting = mmlu_forgetting["E02_noreplay"] - mmlu_forgetting["E11_ssra100"]
forgetting_pct   = delta_forgetting / mmlu_forgetting["E02_noreplay"] * 100

print(f"\n──────────────────────────────────────────")
print(f"SSRA vs no-replay:")
print(f"  Delta MedQA           = {delta_medqa:+.4f}  ({delta_medqa*100:+.1f} pp)")
print(f"  Delta MMLU-forgetting = {-delta_forgetting:.4f}  ({-forgetting_pct:.0f}% reduction)")
print(f"──────────────────────────────────────────")
if delta_medqa >= -0.02:
    print("PASS: SSRA cuts forgetting without meaningful MedQA degradation (<2 pp).")
else:
    print(f"WARN: SSRA loses {abs(delta_medqa):.1%} on MedQA. Report honestly in the paper.")
print(f"──────────────────────────────────────────")

# Values to paste into the master cowork prompt
print(f"\n── PASTE INTO MASTER COWORK PROMPT ──")
print(f"  E02 MedQA test accuracy = {acc_E02:.4f}")
print(f"  E10 MedQA test accuracy = {acc_E10:.4f}")
print(f"  E11 MedQA test accuracy = {acc_E11:.4f}")
print(f"──────────────────────────────────────")

In [ ]:
# ── Cell 10: Pareto frontier plot ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6.5, 4.5))

palette = {"E02_noreplay": "#888888", "E10_real100": "#1f77b4", "E11_ssra100": "#d62728"}
labels  = {"E02_noreplay": "E02 (no replay)", "E10_real100": "E10 (100 real C4)", "E11_ssra100": "E11 (SSRA 100)"}

for cond in df.condition:
    row = df[df.condition == cond].iloc[0]
    ax.scatter(row.MMLU_forgetting, row.MedQA_acc,
               s=160, color=palette[cond], edgecolor="white", linewidths=1.8, zorder=5)
    ax.annotate(labels[cond], (row.MMLU_forgetting, row.MedQA_acc),
                xytext=(10, -4), textcoords="offset points", fontsize=10)

ax.set_xlabel("MMLU mean forgetting  (lower is better)", fontsize=11)
ax.set_ylabel("MedQA test accuracy  (higher is better)", fontsize=11)
ax.set_title(
    f"K0 — Pareto frontier: target accuracy vs general forgetting\n"
    f"Base model: Qwen3.5-9B  |  LoRA r={RANK}  |  {STEPS} steps", fontsize=11)
ax.invert_xaxis()
ax.grid(alpha=0.3, linestyle=":")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(WORK / "k0_pareto.png", dpi=180)
plt.show()
print("Saved: /kaggle/working/k0_pareto.png")
print("Saved: /kaggle/working/k0_pareto_table.csv")

## Reproducibility notes
| Item | Value |
|---|---|
| Base model | `Qwen/Qwen2.5-7B` |
| Architecture | Standard transformer (no DeltaNet hybrid layers) |
| Quantisation | 4-bit NF4 (bitsandbytes) |
| GPU pinning | `device_map={"":0}` (single GPU 0) |
| LoRA rank / alpha | 16 / 16 |
| LoRA target modules | q/k/v/o_proj, gate/up/down_proj |
| Training steps | 500 |
| LR / scheduler | 2e-4 / cosine |
| Batch (per-device / grad-accum / effective) | 1 / 16 / 16 |
| Max sequence length | 512 (via `tok.model_max_length`; question text capped at 1200 chars) |
| Gradient checkpointing | enabled |
| OOM fix | `enable_input_require_grads()` replaces `prepare_model_for_kbit_training()` |
| Seed | 42 |
| MedQA dataset | `GBaker/MedQA-USMLE-4-options-hf` |
| MedQA test slice | 200 examples |
| Replay buffer size | 100 |
| Hardware | Kaggle T4 (16 GB) |